# Riyadh House Price Predictor (مقدّر أسعار الفلل بالرياض)
تحليل وبناء نموذج تعلّم آلة لتقدير أسعار الفلل في الرياض بناءً على إعلانات عقار، مع معايرة مؤشر عقار لسنة 2026.


In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data import split_frame, load_clean
from features import ALL_INPUT_COLS, extract_features
from modeling import candidates, price_metrics, model_input_cols
from aqar_benchmark_mapper import AqarBenchmarkMapper
import joblib

pd.set_option('display.max_columns', 35)


## 1. تدقيق البيانات وحذف التكرارات والتقسيم النظيف
فحص السجلات وحذف التكرار الحرفي والشبه مطابق قبل أي تدريب لتفادي تسريب البيانات (Leakage).


In [2]:
split = split_frame()
train_df, test_df = split.train, split.test
print(f"عدد صفوف التدريب: {len(train_df):,} | عدد صفوف الاختبار (Holdout): {len(test_df):,}")
split.audit.to_frame()


عدد صفوف التدريب: 5,770 | عدد صفوف الاختبار (Holdout): 1,448


,step,rows_before,rows_after,removed,rule
0,raw load,9974,9974,0,file sha256 bac4837af685
1,exact duplicates,9974,7474,2500,"key = location, size, price, bedrooms, bathrooms"
2,price and size bounds,7474,7237,237,"price 150,000-7,000,000 SAR, size 100-1200 m2"
3,price per m2 rule (train),5786,5770,16,"1,000-15,000 SAR/m2"
4,price per m2 rule (test),1451,1448,3,"1,000-15,000 SAR/m2; removed rows kept for a s..."


## 2. تحميل النموذج البطل (Champion Model) واختباره على بيانات الاختبار المحجوبة


In [3]:
artifact = joblib.load('house_price_model.joblib')
pipeline = artifact['pipeline']
cols = artifact['feature_cols']

X_test = test_df[cols]
y_test = np.log(test_df['price'])
pred_log = pipeline.predict(X_test)
pred_price = np.exp(pred_log)

metrics = price_metrics(test_df['price'], pred_price)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:25s}: {v:.4f}" if v < 100 else f"{k:25s}: {v:,.0f}")
    else:
        print(f"{k:25s}: {v}")


n                        : 1448
mean_abs_log_error       : 0.1181
MAPE_pct                 : 12.3416
MdAPE_pct                : 7.3043
within_10pct_share       : 0.6077
within_20pct_share       : 0.8315
median_bias_pct          : -0.0328
R2_log_price             : 0.8387
MAE_SAR                  : 206,154


## 3. تجربة تنبؤ حي ومعايرة أسعار 2026


In [4]:
from inference_example import predict_riyadh_property

res = predict_riyadh_property(
    district="النرجس",
    size=350,
    bedrooms=5,
    bathrooms=5,
    title="فيلا مودرن فاخرة للبيع في النرجس",
    details="فيلا مودرن درج صالة مصعد مسبح تكييف مخفي غرفة خادمة غرفة سائق شارع 20 شمالي"
)
for k, v in res.items():
    print(f"{k:30s}: {v}")


district                      : النرجس


size_m2                       : 350
size_bucket                   : 275-400 m²
estimated_price_2016_sar      : 1975252.621343513
aqar_growth_ratio             : 2.315533005124083
estimated_price_2026_sar      : 4573762.638178768
aqar_benchmark_2026_sar       : 3687440.0
is_district_median_imputed    : False
